In [13]:
# === Imports (один раз в начале ноутбука) ===
import sys, torch
sys.path.append("/mnt/data")
import sys, importlib
from snn_mnist_net import Cfg, run_experiment, save_snn, load_weights_into, build_net,build_encoder_from_cfg
from label_map import build_label_map,save_label_map, load_label_map
from counts_readout import collect_counts_plus_cuda, make_mnist_datasets
from readout_models import tfidf_from_counts, train_mlp_readout
# (опционально) from evaluation import evaluate_on_mnist

_MODS = [
    "snn_mnist_net",
    "label_map",
    "counts_readout",
    "readout_models",
]

# 1) вычистим из sys.modules сам модуль и подпакеты
for m in list(sys.modules):
    if m in _MODS or any(m.startswith(x + ".") for x in _MODS):
        sys.modules.pop(m, None)

# 2) инвалидируем кэши файловой системы
importlib.invalidate_caches()

# 3) заново импортируем модули как модули
import snn_mnist_net, label_map, counts_readout, readout_models

In [14]:


# === FULL RUN CONFIG (адаптировано под Cfg) ===
cfg = Cfg(
    # базовые
    time=200, n_hidden=100, device="cpu", seed=42,
    # обучение
    N=60000,                 # стартовый «полный» прогон
    log_every=500,
    # STDP
    nu_plus=1e-4, nu_minus=-1e-3,
    # ингибиция / WTA
    enable_inhibition_at_start=True,
    top_k=3,
    inhib_strength=0.705,
    # веса
    w_init_lo=0.25, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    # пороги/динамика LIF
    thresh_init=0.38, tau_val=150.0, refrac_val=2.0, reset_val=0.0,
    # кодировщик
    encoder="poisson", poisson_rate_scale=0.011,
    # прочее
    debug=False,
    warmup_N = 100
    
)
print("FULL RUN:", {k: getattr(cfg, k) for k in [
    "inhib_strength","w_init_lo","w_init_hi","poisson_rate_scale","thresh_init","N"
]})




FULL RUN: {'inhib_strength': 0.705, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'poisson_rate_scale': 0.011, 'thresh_init': 0.38, 'N': 60000}


In [ ]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

#save_snn("out/snn_mnist_v5.pt", cfg, FF, lif_layer)

In [ ]:
print("SUMMARY:", summary)

In [15]:
import os
import importlib
import evaluation 
from evaluation import probe_readouts_counts
importlib.invalidate_caches()
importlib.reload(snn_mnist_net)
importlib.reload(evaluation)
importlib.invalidate_caches()
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
enc = build_encoder_from_cfg(cfg)

# 2) загрузить веса и пороги
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")



[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Loaded from out/snn_mnist_v5.pt


In [ ]:
# 3) теперь вызываем build_label_map
label_map_build = build_label_map(net, None, lif_layer, enc, n_calib=2000, T=cfg.time, top_k=3, seed=123)

save_label_map("out/label_map_v5.pt",label_map_build, meta={"ckpt":"out/snn_mnist_v5.pt","T":cfg.time})

In [4]:
label_map_build = load_label_map("out/label_map_v5.pt")

[label_map] loaded: out/label_map_v5.pt  shape=(100,)  meta={'created_at': '2026-01-26 12:18:18', 'ckpt': 'out/snn_mnist_v5.pt', 'T': 200}


In [8]:
from counts_readout import make_mnist_datasets

ds_train, ds_test = make_mnist_datasets()

In [10]:
import importlib

from counts_readout import collect_counts_plus_fast, make_mnist_datasets
from evaluation import probe_readouts_counts, train_mlp_readout
importlib.invalidate_caches()
importlib.reload(counts_readout)
Xtr, ytr, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_train, 60000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,          # было 3.0 → 1.5
    return_debug=True, desc="Collect train: boost=1.5"
)
Xte, yte, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_test, 10000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,
    return_debug=True, desc="Collect test: boost=1.5"
)

N = lif_layer.n
print("mean sum per sample:", float(Xtr[:, :N].sum(1).mean()), float(Xte[:, :N].sum(1).mean()))

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
mean sum per sample: 414.9049377441406 423.0065002441406


In [11]:
N = lif_layer.n
Xtr_n, Xte_n, *_ = tfidf_from_counts(Xtr[:, :N], Xte[:, :N])
_, acc = train_mlp_readout(Xtr_n, ytr, Xte_n, yte, hidden=256, epochs=30, batch_size=256)
print(f"[MLP/TF-IDF] acc = {acc:.3f}")

[MLP/TF-IDF] acc = 0.544


In [7]:
print(dbg_tr)

{'requested_offset': 0.0, 'threshold_abs': None, 'encoder_rate_boost': 3.0, 'temp_disable_inh': False, 'applied_count': 0, 'avg_delta_mean_vt': 0.0, 'restore_ok_fraction': 1.0, 'vt_before_samples': [], 'vt_after_apply_samples': [], 'vt_after_restore_samples': [], 'device': 'cuda', 'batch_size': 64, 'write_pos': 60000, 'expected_batches': 938, 'actual_batches': 938, 'version': 3}


In [ ]:
print("train offset debug:", dbg_tr)

In [ ]:
print(f"[MLP / TF-IDF(counts)] test acc = {acc:.3f}")

In [ ]:
from label_map import load_label_map 
label_map_build = load_label_map ("out/label_map_v5.pt")

In [ ]:
import os
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")

In [16]:
from evaluation import eval_readouts_from_net
enc = build_encoder_from_cfg(cfg)
accs = eval_readouts_from_net(net, lif_layer, enc, cfg, label_map=label_map_build)
print(accs)   # ждём, что хотя бы counts_zscore+Linear или PCAwhiten+Linear > 0.6

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
{'counts_zscore+Linear': 0.4345000088214874, 'PCAwhiten+Linear': 0.42890000343322754, 'TFIDF+MLP': 0.5467000007629395}
